In [ ]:
import sys
import numpy as np
import pandas as pd

# ==========================================
# 1. SETUP SAMPLE DATA USING PANDAS & NUMPY
# ==========================================

# Seed for reproducibility
np.random.seed(42)

# Create a sample database of bank users
data = {
    "Account_No": [1001, 1002, 1003],
    "PIN": ["1234", "5555", "9876"],
    "Name": ["Alice Smith", "Bob Jones", "Charlie Brown"],
    # Using numpy to generate random balances
    "Balance": np.random.choice(
        [2500.50, 50000.00, 150.75], size=3, replace=False
    ),
}

# Load into a Pandas DataFrame using Account_No as the unique key/index
bank_db = pd.DataFrame(data).set_index("Account_No")


# ==========================================
# 2. CORE USSD FUNCTIONS
# ==========================================


def authenticate_user():
    """Simulates the initial USSD dial and login."""
    print("\n--- Welcome to Nova Bank USSD Service ---")
    try:
        acc_num = int(input("Enter your 4-digit Account Number: "))
        pin = input("Enter your 4-digit PIN: ")

        # Check if account exists and PIN matches
        if acc_num in bank_db.index and str(bank_db.loc[acc_num, "PIN"]) == pin:
            return acc_num
        else:
            print("❌ Invalid Account Number or PIN. Access Denied.")
            return None
    except ValueError:
        print("❌ Invalid input. Account number must be digits.")
        return None


def check_balance(acc_num):
    """Fetches and displays the user's balance."""
    balance = bank_db.loc[acc_num, "Balance"]
    print(f"\n💰 Account Balance: ${balance:,.2f}")


def transfer_funds(acc_num):
    """Handles transferring money to another account."""
    try:
        dest_acc = int(input("Enter recipient's Account Number: "))

        if dest_acc == acc_num:
            print("❌ You cannot transfer money to yourself.")
            return

        if dest_acc not in bank_db.index:
            print("❌ Recipient account not found.")
            return

        amount = float(input("Enter amount to transfer: $"))
        current_balance = bank_db.loc[acc_num, "Balance"]

        if amount <= 0:
            print("❌ Amount must be greater than zero.")
        elif amount > current_balance:
            print("❌ Insufficient funds.")
        else:
            # Deduct from sender, add to recipient using Pandas .at
            bank_db.at[acc_num, "Balance"] = current_balance - amount
            bank_db.at[dest_acc, "Balance"] += amount
            print(
                f"✅ Successfully transferred ${amount:,.2f} to {bank_db.loc[dest_acc, 'Name']}."
            )

    except ValueError:
        print("❌ Invalid input. Please enter valid numbers.")


def buy_airtime(acc_num):
    """Deducts balance to simulate purchasing airtime."""
    try:
        phone = input("Enter phone number (e.g., 555-0123): ")
        amount = float(input("Enter airtime amount: $"))
        current_balance = bank_db.loc[acc_num, "Balance"]

        if amount <= 0:
            print("❌ Amount must be greater than zero.")
        elif amount > current_balance:
            print("❌ Insufficient funds.")
        else:
            bank_db.at[acc_num, "Balance"] = current_balance - amount
            print(f"✅ Successfully loaded ${amount:,.2f} of airtime to {phone}.")
    except ValueError:
        print("❌ Invalid input. Please enter a valid amount.")


# ==========================================
# 3. MAIN USSD APPLICATION LOOP
# ==========================================


def main_menu():
    """Simulates the live USSD session loop."""
    # Step 1: Login
    current_user = authenticate_user()

    if not current_user:
        return # Exit if authentication fails

    user_name = bank_db.loc[current_user, "Name"]
    print(f"\nHello, {user_name}!")

    # Step 2: Main Menu Loop
    while True:
        print("\n--- Nova Bank Menu ---")
        print("1. Check Balance")
        print("2. Transfer Money")
        print("3. Buy Airtime")
        print("4. Exit Session")

        choice = input("Select an option (1-4): ").strip()

        if choice == "1":
            check_balance(current_user)
        elif choice == "2":
            transfer_funds(current_user)
        elif choice == "3":
            buy_airtime(current_user)
        elif choice == "4":
            print("\nThank you for banking with Nova Bank. Goodbye!")
            break
        else:
            print("❌ Invalid choice. Please select a valid menu option.")


# Run the application
if __name__ == "__main__":
    # Standard USSD dial code simulation
    dial_code = input("Dial Nova Bank USSD string (e.g., *901#): ")
    if dial_code == "*901#":
        main_menu()
    else:
        print("❌ Connection error: Invalid MMI code.")